# Gait Abnormality Classification on Real PhysioNet Clinical Dataset
###     Machine Learning Mini-Project

This notebook loads and classifies the real clinical human patient recordings from the PhysioNet Gait in Neurodegenerative Disease Database (gaitndd) by Dr. Jeffrey Hausdorff et al. (Harvard Medical School / MIT).

Dataset Cohort (64 Real Patients):
- 16 Healthy Controls (`control1.ts` to `control16.ts`)
- 15 Parkinson's Disease (`park1.ts` to `park15.ts`)
- 20 Huntington's Disease (`hunt1.ts` to `hunt20.ts`)
- 13 ALS (`als1.ts` to `als13.ts`)

In [ ]:
# 1. Hardware & Environment Setup
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Dependencies
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# 3. Download Real Patient Files from PhysioNet Archive
data_dir = 'real_physionet_data'
os.makedirs(data_dir, exist_ok=True)

records = {
    'Control': [f'control{i}' for i in range(1, 17)],
    'Parkinsons': [f'park{i}' for i in range(1, 16)],
    'Huntingtons': [f'hunt{i}' for i in range(1, 21)],
    'ALS': [f'als{i}' for i in range(1, 14)]
}

base_url = 'https://archive.physionet.org/physiobank/database/gaitndd/'
downloaded_files = []

print("Downloading 64 patient time-series records from PhysioNet...")
for condition, file_list in records.items():
    for rec in file_list:
        fname = f"{rec}.ts"
        dest_path = os.path.join(data_dir, fname)
        if not os.path.exists(dest_path):
            url = f"{base_url}{fname}"
            try:
                urllib.request.urlretrieve(url, dest_path)
            except Exception as e:
                print(f"Download error {fname}: {e}")
        downloaded_files.append((dest_path, condition, rec))

print(f"Loaded {len(downloaded_files)} patient records.")

In [ ]:
# 4. Feature Extraction with Clinical Turnaround Filtering
# NOTE: Patients walked back and forth in a 77m hallway. At each turn, stride intervals
# spike abnormally (>2.2s). Filtering turnarounds is essential to avoid corrupting variance.

col_names = [
    'Time', 'Left_Stride', 'Right_Stride', 'Left_Swing', 'Right_Swing',
    'Left_Swing_Pct', 'Right_Swing_Pct', 'Left_Stance', 'Right_Stance',
    'Left_Stance_Pct', 'Right_Stance_Pct', 'Double_Support', 'Double_Support_Pct'
]

subject_features = []

for filepath, condition, rec_id in downloaded_files:
    try:
        df_sub = pd.read_csv(filepath, sep=r'\s+', header=None, names=col_names, engine='python')
        df_sub = df_sub.dropna()

        # Physiological gait filter (exclude hallway turnarounds and sensor disconnects)
        valid_mask = (
            (df_sub['Left_Stride'] >= 0.60) & (df_sub['Left_Stride'] <= 1.80) &
            (df_sub['Right_Stride'] >= 0.60) & (df_sub['Right_Stride'] <= 1.80) &
            (df_sub['Left_Swing'] >= 0.15) & (df_sub['Left_Swing'] <= 0.85) &
            (df_sub['Left_Stance'] >= 0.35) & (df_sub['Left_Stance'] <= 1.30)
        )
        df_clean = df_sub[valid_mask]

        if len(df_clean) < 25:
            continue

        # Sliding window segmentation (25 strides per window, 10 stride step)
        win_size = 25
        step_size = 10
        for start in range(0, len(df_clean) - win_size + 1, step_size):
            win = df_clean.iloc[start:start+win_size]

            m_l = win['Left_Stride'].mean()
            m_r = win['Right_Stride'].mean()
            stride_mean = 0.5 * (m_l + m_r)

            cv_l = (win['Left_Stride'].std() / m_l) * 100.0
            cv_r = (win['Right_Stride'].std() / m_r) * 100.0
            stride_cv = 0.5 * (cv_l + cv_r)

            stride_asym = (abs(m_l - m_r) / (stride_mean + 1e-6)) * 100.0

            sw_l = win['Left_Swing'].mean()
            sw_r = win['Right_Swing'].mean()
            swing_asym = (abs(sw_l - sw_r) / (0.5 * (sw_l + sw_r) + 1e-6)) * 100.0

            st_l = win['Left_Stance'].mean()
            st_r = win['Right_Stance'].mean()
            stance_ratio = (st_l + st_r) / (sw_l + sw_r + 1e-6)

            ds_mean = win['Double_Support'].mean()
            ds_cv = (win['Double_Support'].std() / (ds_mean + 1e-6)) * 100.0

            cadence = 60.0 / (stride_mean + 1e-6)

            subject_features.append({
                'mean_stride': float(stride_mean),
                'stride_cv': float(stride_cv),
                'cadence': float(cadence),
                'stride_asymmetry': float(stride_asym),
                'swing_asymmetry': float(swing_asym),
                'stance_swing_ratio': float(stance_ratio),
                'double_support_mean': float(ds_mean),
                'double_support_cv': float(ds_cv),
                'Subject_ID': rec_id,
                'Condition': condition
            })
    except Exception as err:
        pass

df_features = pd.DataFrame(subject_features)
print(f"Extracted {len(df_features)} cleaned clinical samples across {df_features['Subject_ID'].nunique()} patients.")
print(df_features['Condition'].value_counts())

In [ ]:
# 5. Subject-Disjoint Partitioning & Normalization
classes = sorted(df_features['Condition'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
feature_cols = [c for c in df_features.columns if c not in ['Subject_ID', 'Condition']]

np.random.seed(42)
train_subs, test_subs = [], []
for c in classes:
    subs = df_features[df_features['Condition'] == c]['Subject_ID'].unique()
    np.random.shuffle(subs)
    n_tr = max(1, int(0.70 * len(subs)))
    train_subs.extend(subs[:n_tr])
    test_subs.extend(subs[n_tr:])

train_df = df_features[df_features['Subject_ID'].isin(train_subs)]
test_df = df_features[df_features['Subject_ID'].isin(test_subs)]

X_tr = train_df[feature_cols].values.astype(np.float32)
y_tr = np.array([class_to_idx[c] for c in train_df['Condition']], dtype=np.int64)
X_te = test_df[feature_cols].values.astype(np.float32)
y_te = np.array([class_to_idx[c] for c in test_df['Condition']], dtype=np.int64)

# Robust Standard Scaling
mean = np.mean(X_tr, axis=0, keepdims=True)
std = np.std(X_tr, axis=0, keepdims=True)
std[std == 0] = 1.0
X_tr_sc = (X_tr - mean) / std
X_te_sc = (X_te - mean) / std

print(f"Training set: {len(X_tr)} samples from {len(train_subs)} subjects")
print(f"Testing set:  {len(X_te)} samples from {len(test_subs)} subjects")

In [ ]:
# 6. Train Ensemble Classifier (Random Forest Baseline)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')
rf_model.fit(X_tr_sc, y_tr)

rf_preds = rf_model.predict(X_te_sc)
rf_acc = accuracy_score(y_te, rf_preds)
print(f"Random Forest Test Accuracy: {rf_acc * 100:.2f}%")
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_te, rf_preds, target_names=classes, digits=4))

In [ ]:
# 7. Train Neural Network (Calibrated PyTorch Architecture)
class CalibratedGaitNet(nn.Module):
    def __init__(self, in_features, num_classes):
        super(CalibratedGaitNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 48),
            nn.BatchNorm1d(48),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(48, 24),
            nn.BatchNorm1d(24),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(24, num_classes)
        )
    def forward(self, x):
        return self.net(x)

# Balance class weights to prevent majority-class collapse
class_counts = np.bincount(y_tr)
weights = 1.0 / (class_counts.astype(np.float32) + 1e-6)
weights = weights / np.sum(weights) * len(classes)
class_weights_t = torch.tensor(weights, dtype=torch.float32).to(device)

model = CalibratedGaitNet(len(feature_cols), len(classes)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

train_dataset = TensorDataset(torch.tensor(X_tr_sc, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

epochs = 80
for epoch in range(1, epochs + 1):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        out = model(batch_x)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()

print("Neural Network training complete!")

In [ ]:
# 8. Neural Network Evaluation & Confusion Matrix
model.eval()
with torch.no_grad():
    x_te_t = torch.tensor(X_te_sc, dtype=torch.float32).to(device)
    logits = model(x_te_t)
    nn_preds = torch.argmax(logits, dim=1).cpu().numpy()

nn_acc = accuracy_score(y_te, nn_preds)
print(f"Neural Network Test Accuracy: {nn_acc * 100:.2f}%")
print("\n--- Neural Network Classification Report ---")
print(classification_report(y_te, nn_preds, target_names=classes, digits=4))

# Plot Confusion Matrix
cm = confusion_matrix(y_te, nn_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix (Test Accuracy: {nn_acc*100:.1f}%)')
plt.tight_layout()
plt.show()

In [ ]:
# 9. Live Patient Triage Demo
sample_id = test_subs[0]
patient_row = test_df[test_df['Subject_ID'] == sample_id].iloc[0]
actual_label = patient_row['Condition']

x_sample = np.array([float(patient_row[col]) for col in feature_cols], dtype=np.float32)
x_sample_sc = (x_sample - mean.flatten()) / std.flatten()
x_tensor = torch.tensor(x_sample_sc.reshape(1, -1), dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    logits = model(x_tensor)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = np.argmax(probs)
    pred_class = classes[pred_idx]

print(f"Patient Sample ID: {sample_id}")
print(f"Actual Pathology:  {actual_label}")
print(f"Predicted Class:   {pred_class} (Confidence: {probs[pred_idx]*100:.2f}%)")
print("Probabilities:", {c: round(float(p), 4) for c, p in zip(classes, probs)})
print("\nExtracted Biomarkers:")
for col in feature_cols:
    print(f"  {col:<22}: {float(patient_row[col]):.3f}")